# Notebook 13: Agent Foundations — ReAct & Tool Use from Scratch

**Series**: Frontier ML Interview Prep — Agentic Systems  
**Prerequisites**: Notebook 12 (Prompting & Structured Output), comfortable with Python classes  
**Goal**: Build a complete agent system from scratch — no LangChain, no CrewAI, no frameworks. Interviewers want to see that you understand every layer.

---

## 0. Self-Quiz (Active Recall — attempt BEFORE reading)

Write your answers in the cell below before proceeding. Return here after completing the notebook to check yourself.

1. **What is an agent vs a chatbot?** What are the key structural differences?
2. **What is the ReAct framework?** What does the acronym stand for, and what problem does it solve?
3. **How does function calling work?** What is the difference between text-based tool parsing and structured function calling?
4. **What are the three biggest failure modes of LLM agents?**
5. **Why would an interviewer ask you to build an agent without a framework?**

In [ ]:
# YOUR ANSWERS HERE (fill in before reading the notebook)
self_quiz_answers = {
    "agent_vs_chatbot": "",
    "react_framework": "",
    "function_calling": "",
    "failure_modes": "",
    "why_no_frameworks": "",
}

---
## 1. Setup

In [ ]:
!pip install -q openai torch transformers

In [ ]:
import os
import re
import json
import time
import math
import traceback
from typing import Any, Dict, List, Optional, Callable, Tuple
from dataclasses import dataclass, field
from datetime import datetime
from abc import ABC, abstractmethod
import textwrap

# For Colab: set your API key
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Or set directly:
# os.environ["OPENAI_API_KEY"] = "sk-..."

from openai import OpenAI

# We'll use a flag to allow running without an API key for demonstration
USE_MOCK_LLM = not os.environ.get("OPENAI_API_KEY")
if USE_MOCK_LLM:
    print("No OPENAI_API_KEY found. Running with mock LLM for demonstration.")
    print("Set your API key to use real LLM calls.")
else:
    client = OpenAI()
    print("OpenAI client initialized.")

---
## 2. What Is an Agent?

### The Core Insight

A **chatbot** is a single function call: `response = llm(prompt)`. One input, one output, done.

An **agent** is a **loop**:

```
while not done:
    thought = llm(observations)      # THINK about what to do
    action = select_tool(thought)     # DECIDE which tool to use
    observation = execute(action)     # ACT and observe result
    observations.append(observation)  # REMEMBER what happened
```

The three ingredients:
1. **LLM** — the reasoning engine (decides what to do next)
2. **Tools** — external capabilities (search, calculate, execute code, call APIs)
3. **Loop** — the observe-think-act cycle that continues until the goal is met

### Why This Matters for Interviews

Every frontier lab is building agents. The interviewer wants to know:
- Can you build the core loop from scratch?
- Do you understand the failure modes?
- Can you design robust tool interfaces?
- Do you know when agents are the right solution vs. simpler approaches?

In [ ]:
# Let's make the distinction concrete.

def chatbot(query: str) -> str:
    """A chatbot: one call, one response. No tools, no loop."""
    # In reality: return client.chat.completions.create(...).choices[0].message.content
    return f"I think the answer to '{query}' is ... [single LLM response]"


def agent(query: str, tools: dict, max_steps: int = 10) -> str:
    """An agent: a loop that interleaves thinking and acting."""
    observations = []
    for step in range(max_steps):
        # THINK: what should I do next?
        thought = "llm(query + observations)"  # placeholder
        
        # CHECK: am I done?
        if "Final Answer" in thought:
            return thought
        
        # ACT: use a tool
        tool_name, tool_input = "parse_action(thought)", "..."
        result = "tools[tool_name].execute(tool_input)"  # placeholder
        
        # OBSERVE: record what happened
        observations.append(result)
    
    return "Max steps reached without finding answer."


# The key differences:
comparison = {
    "Feature":          ["Chatbot",           "Agent"],
    "LLM calls":        ["1",                 "N (dynamic)"],
    "Tools":            ["None",              "Multiple, composable"],
    "State":            ["Stateless",         "Maintains observations"],
    "Reasoning":        ["Single-shot",       "Multi-step, iterative"],
    "Error recovery":   ["None",              "Can retry, re-plan"],
    "Controllability":  ["High",              "Lower (emergent behavior)"],
    "Cost":             ["Low (1 API call)",  "Higher (N API calls)"],
}

# Pretty print
print(f"{'Feature':<20} {'Chatbot':<25} {'Agent':<30}")
print("-" * 75)
for feature, (chatbot_val, agent_val) in comparison.items():
    if feature != "Feature":
        print(f"{feature:<20} {chatbot_val:<25} {agent_val:<30}")

**Insider Tip:** In interviews at Anthropic or OpenAI, you'll be asked to build an agent from scratch WITHOUT frameworks. LangChain/CrewAI are fine for prototyping, but interviewers want to see you understand the internals: prompt construction, output parsing, tool routing, error recovery. This notebook is exactly what they expect.

---
## 3. Tool Abstraction

Before we build an agent, we need tools for it to use. A good tool abstraction needs:
- **Name**: how the agent refers to it
- **Description**: what the tool does (the LLM reads this to decide when to use it)
- **Parameters**: JSON schema describing the expected input
- **Execute method**: actually runs the tool and returns output

This is the exact same interface that OpenAI/Anthropic use for function calling.

In [ ]:
class Tool(ABC):
    """Abstract base class for all agent tools.
    
    Every tool must define:
    - name: identifier the LLM uses to call this tool
    - description: natural language description (the LLM reads this!)
    - parameters: JSON schema for the tool's input
    - execute(): the actual implementation
    """
    
    @property
    @abstractmethod
    def name(self) -> str:
        pass
    
    @property
    @abstractmethod
    def description(self) -> str:
        pass
    
    @property
    @abstractmethod
    def parameters(self) -> Dict[str, Any]:
        """JSON Schema describing the tool's input parameters."""
        pass
    
    @abstractmethod
    def execute(self, **kwargs) -> str:
        """Execute the tool and return a string result."""
        pass
    
    def to_openai_function(self) -> Dict[str, Any]:
        """Convert to OpenAI function calling format."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            }
        }
    
    def format_for_prompt(self) -> str:
        """Format tool description for inclusion in a text prompt."""
        params_desc = []
        props = self.parameters.get("properties", {})
        required = self.parameters.get("required", [])
        for param_name, param_info in props.items():
            req = "(required)" if param_name in required else "(optional)"
            params_desc.append(
                f"    - {param_name} ({param_info.get('type', 'string')}): "
                f"{param_info.get('description', '')} {req}"
            )
        params_str = "\n".join(params_desc) if params_desc else "    (no parameters)"
        return f"Tool: {self.name}\nDescription: {self.description}\nParameters:\n{params_str}"


print("Tool base class defined.")

In [ ]:
class CalculatorTool(Tool):
    """Safely evaluates mathematical expressions.
    
    Security note: We use a restricted eval with only math functions
    available. In production, you'd use a proper math parser.
    """
    
    @property
    def name(self) -> str:
        return "Calculator"
    
    @property
    def description(self) -> str:
        return (
            "Evaluates a mathematical expression and returns the result. "
            "Supports basic arithmetic (+, -, *, /, **), math functions "
            "(sqrt, sin, cos, log, etc.), and constants (pi, e). "
            "Input should be a valid Python math expression."
        )
    
    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate, e.g. '2 + 2' or 'sqrt(144)'"
                }
            },
            "required": ["expression"]
        }
    
    def execute(self, expression: str = "", **kwargs) -> str:
        """Safely evaluate a math expression."""
        # Whitelist of allowed names — only math functions and constants
        allowed_names = {
            k: v for k, v in math.__dict__.items()
            if not k.startswith('_')
        }
        allowed_names.update({
            'abs': abs, 'round': round, 'min': min, 'max': max,
            'int': int, 'float': float,
        })
        
        try:
            # Security: no builtins, only math functions
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            return str(result)
        except Exception as e:
            return f"Error evaluating '{expression}': {type(e).__name__}: {e}"


# Test it
calc = CalculatorTool()
print(calc.format_for_prompt())
print("\n--- Tests ---")
print(f"2 + 2 = {calc.execute(expression='2 + 2')}")
print(f"sqrt(144) = {calc.execute(expression='sqrt(144)')}")
print(f"pi * 3**2 = {calc.execute(expression='pi * 3**2')}")
print(f"log(e) = {calc.execute(expression='log(e)')}")
print(f"Bad input: {calc.execute(expression='import os')}")

In [ ]:
class SearchTool(Tool):
    """Simulated web search tool.
    
    Uses a mock knowledge base for demonstration. In production,
    you'd connect this to a real search API (Google, Bing, Tavily, etc.)
    """
    
    def __init__(self):
        # Mock knowledge base for demonstration
        self._knowledge = {
            "GDP France": "France's GDP in 2024 was approximately $3.13 trillion USD, making it the 7th largest economy in the world.",
            "GDP Germany": "Germany's GDP in 2024 was approximately $4.46 trillion USD, making it the 3rd largest economy in the world.",
            "population Japan": "Japan's population in 2024 was approximately 123.3 million people, and has been declining since 2010.",
            "RLHF": "Reinforcement Learning from Human Feedback (RLHF) is a technique for fine-tuning LLMs using human preference data. Key papers: InstructGPT (Ouyang et al. 2022), Constitutional AI (Bai et al. 2022).",
            "ReAct paper": "ReAct: Synergizing Reasoning and Acting in Language Models (Yao et al. 2022). Key insight: interleaving reasoning (chain-of-thought) and acting (tool use) improves both over using either alone. Published at ICLR 2023.",
            "transformer architecture": "The Transformer architecture (Vaswani et al. 2017) uses self-attention to process sequences in parallel. Key components: multi-head attention, positional encoding, layer normalization, feed-forward networks.",
            "attention mechanism": "Attention computes a weighted sum of values based on query-key similarity: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V. This allows the model to focus on relevant parts of the input.",
            "multi-agent systems": "Multi-agent systems involve multiple AI agents collaborating or competing. Key frameworks: debate, division of labor, hierarchical planning. Challenges: coordination, communication, credit assignment.",
            "role-playing agents": "Role-playing frameworks assign agents distinct personas that interact to solve tasks cooperatively. Key insight: structured prompting guides each agent's behavior.",
            "Python programming": "Python is a high-level programming language known for its readability. Version 3.12 was released in October 2023. Popular for ML/AI due to libraries like NumPy, PyTorch, and TensorFlow.",
            "quantum computing": "Quantum computing uses quantum mechanical phenomena (superposition, entanglement) to perform computation. Current state: 1000+ qubit processors exist but error rates limit practical applications.",
            "climate change": "Global average temperature has risen approximately 1.1 degrees C above pre-industrial levels. Paris Agreement targets limiting warming to 1.5 degrees C. Current trajectory: 2.5-3 degrees C by 2100.",
        }
    
    @property
    def name(self) -> str:
        return "Search"
    
    @property
    def description(self) -> str:
        return (
            "Searches the web for information on a given query. "
            "Returns relevant search results as text. "
            "Use this to find factual information, current data, or research topics."
        )
    
    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query"
                }
            },
            "required": ["query"]
        }
    
    def execute(self, query: str = "", **kwargs) -> str:
        """Search the mock knowledge base."""
        query_lower = query.lower()
        results = []
        for key, value in self._knowledge.items():
            # Simple keyword matching
            if any(word in query_lower for word in key.lower().split()):
                results.append(value)
        
        if results:
            return "\n\n".join(results[:3])  # Return top 3 matches
        else:
            return f"No results found for '{query}'. Try different search terms."

# Test it
search = SearchTool()
print("--- Search Tests ---")
print(f"Query 'GDP France': {search.execute(query='GDP France')}")
print(f"\nQuery 'ReAct paper': {search.execute(query='ReAct paper')}")
print(f"\nQuery 'nonexistent topic xyz': {search.execute(query='nonexistent topic xyz')}")

In [ ]:
class PythonREPLTool(Tool):
    """Executes Python code in a sandboxed environment.
    
    Security note: This uses a restricted exec() with limited builtins.
    In production, you'd use a proper sandbox (Docker, gVisor, etc.)
    """
    
    @property
    def name(self) -> str:
        return "PythonREPL"
    
    @property
    def description(self) -> str:
        return (
            "Executes Python code and returns the output. "
            "The code runs in a sandboxed environment with access to math, "
            "json, and basic Python builtins. Use print() to produce output. "
            "Variables persist between calls within the same session."
        )
    
    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "The Python code to execute"
                }
            },
            "required": ["code"]
        }
    
    def __init__(self):
        # Persistent namespace for the REPL session
        self._namespace = {
            "__builtins__": {
                "print": print, "len": len, "range": range, "enumerate": enumerate,
                "zip": zip, "map": map, "filter": filter, "sorted": sorted,
                "reversed": reversed, "list": list, "dict": dict, "set": set,
                "tuple": tuple, "str": str, "int": int, "float": float,
                "bool": bool, "sum": sum, "min": min, "max": max,
                "abs": abs, "round": round, "isinstance": isinstance,
                "type": type, "hasattr": hasattr, "getattr": getattr,
                "ValueError": ValueError, "TypeError": TypeError,
                "KeyError": KeyError, "IndexError": IndexError,
                "Exception": Exception, "True": True, "False": False,
                "None": None,
            },
            "math": math,
            "json": json,
        }
    
    def execute(self, code: str = "", **kwargs) -> str:
        """Execute Python code in a sandboxed namespace."""
        import io
        import sys
        
        # Capture stdout
        old_stdout = sys.stdout
        sys.stdout = captured = io.StringIO()
        
        try:
            exec(code, self._namespace)
            output = captured.getvalue()
            if not output:
                # Try to get the value of the last expression
                try:
                    result = eval(code.strip().split('\n')[-1], self._namespace)
                    if result is not None:
                        output = str(result)
                except:
                    output = "(Code executed successfully, no output)"
            return output.strip()
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"
        finally:
            sys.stdout = old_stdout


# Test it
repl = PythonREPLTool()
print("--- Python REPL Tests ---")
print(repl.execute(code="print('Hello, world!')"))
print(repl.execute(code="x = [1, 2, 3, 4, 5]\nprint(f'Sum: {sum(x)}, Mean: {sum(x)/len(x)}')" ))
print(repl.execute(code="print(x)"))  # x persists from previous call
print(repl.execute(code="1/0"))  # Error handling

In [ ]:
class ToolRegistry:
    """Registry that manages available tools for an agent.
    
    Responsibilities:
    - Register tools by name
    - Look up tools by name
    - Format all tool descriptions for the LLM prompt
    - Convert to OpenAI function calling format
    """
    
    def __init__(self):
        self._tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool) -> None:
        """Register a tool."""
        self._tools[tool.name] = tool
    
    def get(self, name: str) -> Optional[Tool]:
        """Get a tool by name (case-insensitive)."""
        # Try exact match first, then case-insensitive
        if name in self._tools:
            return self._tools[name]
        for tool_name, tool in self._tools.items():
            if tool_name.lower() == name.lower():
                return tool
        return None
    
    def list_tools(self) -> List[str]:
        """List all registered tool names."""
        return list(self._tools.keys())
    
    def format_for_prompt(self) -> str:
        """Format all tools for inclusion in a text prompt."""
        parts = []
        for tool in self._tools.values():
            parts.append(tool.format_for_prompt())
        return "\n\n".join(parts)
    
    def to_openai_functions(self) -> List[Dict[str, Any]]:
        """Convert all tools to OpenAI function calling format."""
        return [tool.to_openai_function() for tool in self._tools.values()]


# Create a registry with our tools
registry = ToolRegistry()
registry.register(CalculatorTool())
registry.register(SearchTool())
registry.register(PythonREPLTool())

print("Registered tools:", registry.list_tools())
print("\n" + "=" * 60)
print("Tools formatted for prompt:")
print("=" * 60)
print(registry.format_for_prompt())

---
## 4. ReAct Agent from Scratch

**ReAct** (Reasoning + Acting) is the foundational agent architecture. The key insight from Yao et al. (2022):

> Interleaving reasoning traces and actions allows the model to:
> 1. **Create plans** and track progress (reasoning guides acting)
> 2. **Ground reasoning** in real observations (acting grounds reasoning)

The format:
```
Thought: [reasoning about what to do]
Action: [tool name]
Action Input: [input to the tool]
Observation: [result from the tool]
... (repeat)
Thought: [final reasoning]
Final Answer: [the answer]
```

Let's build it.

In [ ]:
class ReActAgent:
    """A ReAct agent that interleaves reasoning and acting.
    
    Built from scratch — no frameworks. This is what interviewers want to see.
    
    Architecture:
    1. System prompt describes available tools and ReAct format
    2. User query starts the loop
    3. LLM generates Thought + Action + Action Input
    4. Agent parses the response, executes the tool
    5. Observation is appended to the conversation
    6. Loop continues until "Final Answer" or max_steps
    """
    
    def __init__(
        self,
        model: str = "gpt-4o-mini",
        tools: Optional[List[Tool]] = None,
        max_steps: int = 10,
        verbose: bool = True,
    ):
        self.model = model
        self.max_steps = max_steps
        self.verbose = verbose
        
        # Set up tool registry
        self.registry = ToolRegistry()
        if tools:
            for tool in tools:
                self.registry.register(tool)
        
        # LLM client
        self.client = None if USE_MOCK_LLM else OpenAI()
    
    def _build_system_prompt(self) -> str:
        """Build the system prompt with tool descriptions and ReAct format."""
        tool_descriptions = self.registry.format_for_prompt()
        
        return f"""You are a helpful assistant that can use tools to answer questions.

You have access to the following tools:

{tool_descriptions}

To use a tool, you MUST follow this EXACT format:

Thought: [your reasoning about what to do next]
Action: [the tool name, exactly as listed above]
Action Input: [the input to the tool]

After each tool use, you will receive an Observation with the result.
You can then continue with another Thought/Action/Action Input, or provide your final answer.

When you have enough information to answer the question, respond with:

Thought: [your final reasoning]
Final Answer: [your complete answer to the user's question]

IMPORTANT:
- Always start with a Thought before taking an action
- Use the exact tool names as listed above
- Only use one tool per step
- Be concise in your thoughts
- If a tool returns an error, try a different approach"""
    
    def _parse_response(self, text: str) -> Dict[str, Optional[str]]:
        """Parse the LLM response to extract Thought, Action, Action Input, or Final Answer.
        
        Returns:
            dict with keys: 'thought', 'action', 'action_input', 'final_answer'
        """
        result = {
            'thought': None,
            'action': None,
            'action_input': None,
            'final_answer': None,
        }
        
        # Extract Thought
        thought_match = re.search(r'Thought:\s*(.+?)(?=\n(?:Action|Final Answer)|$)', text, re.DOTALL)
        if thought_match:
            result['thought'] = thought_match.group(1).strip()
        
        # Check for Final Answer
        final_match = re.search(r'Final Answer:\s*(.+)', text, re.DOTALL)
        if final_match:
            result['final_answer'] = final_match.group(1).strip()
            return result
        
        # Extract Action and Action Input
        action_match = re.search(r'Action:\s*(.+?)(?=\n|$)', text)
        if action_match:
            result['action'] = action_match.group(1).strip()
        
        input_match = re.search(r'Action Input:\s*(.+?)(?=\n(?:Thought|Observation)|$)', text, re.DOTALL)
        if input_match:
            result['action_input'] = input_match.group(1).strip()
        
        return result
    
    def _execute_tool(self, action: str, action_input: str) -> str:
        """Execute a tool by name and return the result."""
        tool = self.registry.get(action)
        if tool is None:
            available = ", ".join(self.registry.list_tools())
            return f"Error: Tool '{action}' not found. Available tools: {available}"
        
        try:
            # Determine the parameter name from the tool's schema
            props = tool.parameters.get("properties", {})
            param_names = list(props.keys())
            
            if len(param_names) == 1:
                # Single parameter — pass action_input directly
                return tool.execute(**{param_names[0]: action_input})
            else:
                # Multiple parameters — try to parse as JSON
                try:
                    params = json.loads(action_input)
                    return tool.execute(**params)
                except json.JSONDecodeError:
                    return tool.execute(**{param_names[0]: action_input})
        except Exception as e:
            return f"Error executing tool '{action}': {type(e).__name__}: {e}"
    
    def _call_llm(self, messages: List[Dict[str, str]]) -> str:
        """Call the LLM and return the response text."""
        if USE_MOCK_LLM:
            return self._mock_llm_response(messages)
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0,
            max_tokens=1024,
        )
        return response.choices[0].message.content
    
    def _mock_llm_response(self, messages: List[Dict[str, str]]) -> str:
        """Generate mock LLM responses for demonstration without API key."""
        last_msg = messages[-1]["content"] if messages else ""
        user_msgs = [m["content"] for m in messages if m["role"] == "user"]
        user_query = user_msgs[0] if user_msgs else ""
        
        # Determine step based on number of observations in conversation
        observation_count = sum(1 for m in messages if "Observation:" in m.get("content", ""))
        
        if "GDP" in user_query and "France" in user_query and "Germany" in user_query:
            responses = [
                "Thought: I need to find the GDP of France and Germany. Let me search for France's GDP first.\nAction: Search\nAction Input: GDP France",
                "Thought: I found France's GDP is $3.13 trillion. Now let me search for Germany's GDP.\nAction: Search\nAction Input: GDP Germany",
                "Thought: Germany's GDP is $4.46 trillion. Now I can calculate the ratio.\nAction: Calculator\nAction Input: 4.46 / 3.13",
                "Thought: The ratio of Germany's GDP to France's GDP is approximately 1.425. I now have all the information needed.\nFinal Answer: Germany's GDP ($4.46 trillion) is approximately 1.42 times larger than France's GDP ($3.13 trillion). The ratio is about 1.425:1."
            ]
        elif "fibonacci" in user_query.lower():
            responses = [
                "Thought: I need to write Python code to compute the 20th Fibonacci number.\nAction: PythonREPL\nAction Input: def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\nprint(f'The 20th Fibonacci number is: {fib(20)}')",
                "Thought: The code executed and gave me the result.\nFinal Answer: The 20th Fibonacci number is 6765."
            ]
        elif "ReAct" in user_query or "react" in user_query.lower():
            responses = [
                "Thought: Let me search for information about the ReAct paper.\nAction: Search\nAction Input: ReAct paper",
                "Thought: I found the information. Now let me also search for how it relates to the transformer architecture.\nAction: Search\nAction Input: transformer architecture",
                "Thought: I now have comprehensive information to answer the question.\nFinal Answer: ReAct (Reasoning + Acting) is a framework by Yao et al. (2022) that interleaves reasoning traces (chain-of-thought) with actions (tool use). Published at ICLR 2023, its key insight is that reasoning alone lacks grounding in real-world observations, while acting alone lacks the planning capability that reasoning provides. By combining both, agents can create plans, track progress, handle exceptions, and ground their reasoning in actual observations."
            ]
        else:
            responses = [
                "Thought: Let me search for information on this topic.\nAction: Search\nAction Input: " + user_query,
                "Thought: I have enough information to answer.\nFinal Answer: Based on my research, here is the answer to your question about '" + user_query + "'."
            ]
        
        idx = min(observation_count, len(responses) - 1)
        return responses[idx]
    
    def run(self, query: str) -> str:
        """Run the agent on a query.
        
        This is the main loop:
        1. Send query + history to LLM
        2. Parse response for Thought/Action or Final Answer
        3. If Final Answer, return it
        4. If Action, execute the tool and add Observation
        5. Repeat until done or max_steps reached
        """
        messages = [
            {"role": "system", "content": self._build_system_prompt()},
            {"role": "user", "content": query},
        ]
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"Query: {query}")
            print(f"{'='*60}")
        
        for step in range(self.max_steps):
            # Get LLM response
            response_text = self._call_llm(messages)
            
            # Parse the response
            parsed = self._parse_response(response_text)
            
            if self.verbose:
                print(f"\n--- Step {step + 1} ---")
                if parsed['thought']:
                    print(f"Thought: {parsed['thought']}")
            
            # Check for Final Answer
            if parsed['final_answer']:
                if self.verbose:
                    print(f"Final Answer: {parsed['final_answer']}")
                return parsed['final_answer']
            
            # Execute action
            if parsed['action'] and parsed['action_input'] is not None:
                if self.verbose:
                    print(f"Action: {parsed['action']}")
                    print(f"Action Input: {parsed['action_input']}")
                
                observation = self._execute_tool(parsed['action'], parsed['action_input'])
                
                if self.verbose:
                    print(f"Observation: {observation}")
                
                # Add the full exchange to messages
                messages.append({"role": "assistant", "content": response_text})
                messages.append({"role": "user", "content": f"Observation: {observation}"})
            else:
                # Parsing failed — re-prompt
                if self.verbose:
                    print(f"Parse error: Could not extract action from response")
                    print(f"Raw response: {response_text[:200]}...")
                
                messages.append({"role": "assistant", "content": response_text})
                messages.append({
                    "role": "user",
                    "content": (
                        "I couldn't parse your response. Please use the exact format:\n"
                        "Thought: [your reasoning]\n"
                        "Action: [tool name]\n"
                        "Action Input: [tool input]\n\n"
                        "Or if you're done:\n"
                        "Thought: [reasoning]\n"
                        "Final Answer: [your answer]"
                    )
                })
        
        return "Agent reached maximum steps without finding an answer."


print("ReActAgent class defined.")

In [ ]:
# Example 1: Multi-step research with search + calculator
agent = ReActAgent(
    tools=[CalculatorTool(), SearchTool(), PythonREPLTool()],
    verbose=True,
)

result = agent.run(
    "What is the ratio of Germany's GDP to France's GDP? "
    "Search for both values and then calculate the ratio."
)

In [ ]:
# Example 2: Code execution
result = agent.run("What is the 20th Fibonacci number? Write Python code to compute it.")

In [ ]:
# Example 3: Research question
result = agent.run("What is the ReAct framework and why is it important for AI agents?")

---
## 5. Function Calling (Structured Tool Use)

ReAct's text parsing is brittle. The modern approach: **function calling** — the LLM outputs structured JSON describing which tool to call and with what arguments.

### Text parsing vs Function calling

| Aspect | ReAct (text parsing) | Function calling |
|--------|---------------------|------------------|
| Format | Free-text with regex parsing | Structured JSON |
| Reliability | ~85-95% parse success | ~99%+ (model-enforced) |
| Supported by | Any LLM | OpenAI, Anthropic, Google, etc. |
| Flexibility | Can express anything | Constrained to schema |
| Debugging | Hard (regex failures) | Easy (JSON validation) |
| Training | Not specially trained | Models trained for this format |

### How it works

**OpenAI format**: You pass `tools=[{"type": "function", "function": {...}}]` to the API. The model responds with `tool_calls` containing the function name and JSON arguments.

**Anthropic format**: You pass `tools=[{"name": ..., "description": ..., "input_schema": {...}}]`. The model responds with `tool_use` content blocks.

Both achieve the same thing: structured, reliable tool invocation.

**Insider Tip:** MCP (Model Context Protocol), introduced by Anthropic in November 2024, became the industry standard for tool integration and moved to Linux Foundation ecosystem governance in December 2025. It provides standardized tool schemas, auth handling, and transport. Think of it as "USB for AI tools". If you're interviewing at Anthropic, knowing MCP is a strong signal.

In [ ]:
class FunctionCallingAgent:
    """Agent that uses structured function calling instead of text parsing.
    
    This is the modern approach used by OpenAI, Anthropic, and Google.
    Key advantage: no regex parsing, structured JSON, higher reliability.
    """
    
    def __init__(
        self,
        model: str = "gpt-4o-mini",
        tools: Optional[List[Tool]] = None,
        max_steps: int = 10,
        verbose: bool = True,
    ):
        self.model = model
        self.max_steps = max_steps
        self.verbose = verbose
        
        self.registry = ToolRegistry()
        if tools:
            for tool in tools:
                self.registry.register(tool)
        
        self.client = None if USE_MOCK_LLM else OpenAI()
    
    def _execute_tool_call(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        """Execute a function call and return the result."""
        tool = self.registry.get(tool_name)
        if tool is None:
            return f"Error: Tool '{tool_name}' not found."
        
        try:
            return tool.execute(**arguments)
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"
    
    def run(self, query: str) -> str:
        """Run the function-calling agent."""
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use the provided tools to "
                    "answer the user's question. Think step by step."
                )
            },
            {"role": "user", "content": query}
        ]
        
        openai_tools = self.registry.to_openai_functions()
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"[FunctionCallingAgent] Query: {query}")
            print(f"{'='*60}")
        
        if USE_MOCK_LLM:
            # Demonstrate the flow without API calls
            print("\n[Mock mode] Demonstrating function calling flow:")
            print(f"  Tools provided: {json.dumps([t['function']['name'] for t in openai_tools])}")
            
            # Simulate a tool call
            mock_tool_call = {
                "name": "Calculator",
                "arguments": {"expression": "42 * 2"}
            }
            print(f"  Model would return tool_call: {json.dumps(mock_tool_call)}")
            result = self._execute_tool_call(
                mock_tool_call["name"],
                mock_tool_call["arguments"]
            )
            print(f"  Tool result: {result}")
            print(f"  Model would then generate final answer based on result.")
            return f"[Mock] Result: {result}"
        
        for step in range(self.max_steps):
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto",
                temperature=0,
            )
            
            message = response.choices[0].message
            
            # Check if model wants to call tools
            if message.tool_calls:
                messages.append(message)  # Add assistant message with tool calls
                
                for tool_call in message.tool_calls:
                    fn_name = tool_call.function.name
                    fn_args = json.loads(tool_call.function.arguments)
                    
                    if self.verbose:
                        print(f"\n  Step {step + 1}: Calling {fn_name}({fn_args})")
                    
                    result = self._execute_tool_call(fn_name, fn_args)
                    
                    if self.verbose:
                        print(f"  Result: {result}")
                    
                    # Add tool result to messages
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    })
            else:
                # Model returned a text response (no more tool calls)
                if self.verbose:
                    print(f"\n  Final Answer: {message.content}")
                return message.content
        
        return "Max steps reached."


# Demo
fc_agent = FunctionCallingAgent(
    tools=[CalculatorTool(), SearchTool(), PythonREPLTool()],
    verbose=True,
)
fc_agent.run("What is 42 * 2?")

In [ ]:
# Side-by-side comparison of the two approaches

print("COMPARISON: ReAct Text Parsing vs Function Calling")
print("=" * 60)

# ReAct: what the LLM outputs
react_output = """Thought: I need to calculate 42 * 2
Action: Calculator
Action Input: 42 * 2"""

# Function calling: what the LLM outputs (structured)
fc_output = {
    "tool_calls": [{
        "id": "call_abc123",
        "type": "function",
        "function": {
            "name": "Calculator",
            "arguments": '{"expression": "42 * 2"}'
        }
    }]
}

print("\n--- ReAct (text parsing) ---")
print(react_output)
print("\nParsing required: regex to extract Action and Action Input")
print("Failure mode: LLM writes 'Action:calculator' instead of 'Action: Calculator'")

print("\n--- Function Calling (structured) ---")
print(json.dumps(fc_output, indent=2))
print("\nParsing required: json.loads() — well-defined, reliable")
print("Failure mode: almost none — model is trained to output valid JSON")

print("\n--- When to use which ---")
print("ReAct: when using open-source models without function calling support")
print("Function calling: whenever available — it's strictly more reliable")
print("Interview: know BOTH — understanding ReAct shows depth")

---
## 6. Error Handling & Robustness

Real agents fail. A lot. The difference between a demo and a production agent is **error handling**.

Three categories of failure:
1. **Tool failures**: tool throws an exception, returns error, times out
2. **Parse failures**: LLM output doesn't match expected format
3. **Logic failures**: agent loops forever, picks wrong tool, hallucinates tool calls

Let's build a robust agent that handles all three.

**Insider Tip:** The biggest real-world challenge with agents isn't the LLM -- it's reliability. Tool calls fail, parsing breaks, context windows overflow. Production agents need extensive error handling, retry logic, and graceful degradation. Show this awareness in interviews.

In [ ]:
@dataclass
class AgentStep:
    """Records a single step of agent execution for tracing."""
    step_number: int
    thought: Optional[str] = None
    action: Optional[str] = None
    action_input: Optional[str] = None
    observation: Optional[str] = None
    error: Optional[str] = None
    duration_ms: float = 0.0
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class AgentTrace:
    """Complete execution trace of an agent run."""
    query: str
    steps: List[AgentStep] = field(default_factory=list)
    final_answer: Optional[str] = None
    total_duration_ms: float = 0.0
    total_llm_calls: int = 0
    total_tool_calls: int = 0
    errors: List[str] = field(default_factory=list)
    
    def add_step(self, step: AgentStep):
        self.steps.append(step)
        if step.action:
            self.total_tool_calls += 1
        if step.error:
            self.errors.append(step.error)
        self.total_llm_calls += 1
    
    def display(self):
        """Pretty-print the execution trace."""
        print(f"\n{'='*70}")
        print(f"AGENT TRACE")
        print(f"{'='*70}")
        print(f"Query: {self.query}")
        print(f"Total steps: {len(self.steps)}")
        print(f"LLM calls: {self.total_llm_calls}")
        print(f"Tool calls: {self.total_tool_calls}")
        print(f"Errors: {len(self.errors)}")
        print(f"Total duration: {self.total_duration_ms:.0f}ms")
        print(f"{'-'*70}")
        
        for step in self.steps:
            print(f"\n  Step {step.step_number} ({step.duration_ms:.0f}ms):")
            if step.thought:
                print(f"    Thought: {step.thought[:100]}{'...' if len(step.thought or '') > 100 else ''}")
            if step.action:
                print(f"    Action: {step.action}")
                print(f"    Input: {(step.action_input or '')[:80]}{'...' if len(step.action_input or '') > 80 else ''}")
            if step.observation:
                print(f"    Observation: {step.observation[:100]}{'...' if len(step.observation or '') > 100 else ''}")
            if step.error:
                print(f"    ERROR: {step.error}")
        
        print(f"\n{'-'*70}")
        if self.final_answer:
            print(f"Final Answer: {self.final_answer}")
        print(f"{'='*70}")


print("AgentStep and AgentTrace defined.")

In [ ]:
class RobustReActAgent(ReActAgent):
    """ReAct agent with comprehensive error handling.
    
    Extends ReActAgent with:
    - Retry with backoff on tool failures
    - Re-prompt on parse failures
    - Loop detection (repeating the same action)
    - Timeout warning (advisory: flags slow tools after they return; does not kill them)
    - Full execution tracing
    """
    
    def __init__(
        self,
        model: str = "gpt-4o-mini",
        tools: Optional[List[Tool]] = None,
        max_steps: int = 10,
        max_retries: int = 3,
        tool_timeout: float = 30.0,
        verbose: bool = True,
    ):
        super().__init__(model=model, tools=tools, max_steps=max_steps, verbose=verbose)
        self.max_retries = max_retries
        self.tool_timeout = tool_timeout
        self._action_history: List[Tuple[str, str]] = []
    
    def _detect_loop(self, action: str, action_input: str) -> bool:
        """Detect if the agent is repeating the same action."""
        current = (action, action_input)
        # Check if this exact action+input appeared in the last 3 steps
        recent = self._action_history[-3:]
        count = sum(1 for a in recent if a == current)
        return count >= 2  # Same action called 2+ times in last 3 steps
    
    def _execute_with_retry(self, action: str, action_input: str) -> str:
        """Execute a tool with retry logic.

        Note: tool_timeout is advisory only -- we warn if a tool ran longer
        than the limit after it returns; the call is never killed.
        """
        for attempt in range(self.max_retries):
            try:
                start = time.time()
                result = self._execute_tool(action, action_input)
                elapsed = time.time() - start
                
                if elapsed > self.tool_timeout:
                    return f"Warning: Tool '{action}' took {elapsed:.1f}s (timeout={self.tool_timeout}s). Result: {result}"
                
                # Check if result indicates an error worth retrying
                if result.startswith("Error:") and attempt < self.max_retries - 1:
                    backoff = 2 ** attempt * 0.1  # 0.1s, 0.2s, 0.4s
                    if self.verbose:
                        print(f"    [Retry {attempt + 1}/{self.max_retries}] "
                              f"Tool error, retrying in {backoff:.1f}s: {result[:100]}")
                    time.sleep(backoff)
                    continue
                
                return result
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    backoff = 2 ** attempt * 0.1
                    time.sleep(backoff)
                    continue
                return f"Error after {self.max_retries} retries: {type(e).__name__}: {e}"
        
        return "Error: max retries exceeded"
    
    def run(self, query: str) -> Tuple[str, AgentTrace]:
        """Run the robust agent with full tracing.
        
        Returns:
            Tuple of (answer, trace)
        """
        trace = AgentTrace(query=query)
        self._action_history = []
        run_start = time.time()
        
        messages = [
            {"role": "system", "content": self._build_system_prompt()},
            {"role": "user", "content": query},
        ]
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"[RobustReActAgent] Query: {query}")
            print(f"{'='*60}")
        
        parse_failures = 0
        max_parse_failures = 3
        
        for step_num in range(1, self.max_steps + 1):
            step_start = time.time()
            step = AgentStep(step_number=step_num)
            
            # Get LLM response
            try:
                response_text = self._call_llm(messages)
            except Exception as e:
                step.error = f"LLM call failed: {type(e).__name__}: {e}"
                step.duration_ms = (time.time() - step_start) * 1000
                trace.add_step(step)
                if self.verbose:
                    print(f"\n  Step {step_num}: LLM ERROR: {step.error}")
                break
            
            # Parse response
            parsed = self._parse_response(response_text)
            step.thought = parsed['thought']
            
            if self.verbose:
                print(f"\n--- Step {step_num} ---")
                if step.thought:
                    print(f"  Thought: {step.thought}")
            
            # Check for Final Answer
            if parsed['final_answer']:
                trace.final_answer = parsed['final_answer']
                step.duration_ms = (time.time() - step_start) * 1000
                trace.add_step(step)
                if self.verbose:
                    print(f"  Final Answer: {parsed['final_answer'][:200]}")
                break
            
            # Handle parse failure
            if not parsed['action'] or parsed['action_input'] is None:
                parse_failures += 1
                step.error = f"Parse failure #{parse_failures}"
                step.duration_ms = (time.time() - step_start) * 1000
                trace.add_step(step)
                
                if parse_failures >= max_parse_failures:
                    trace.final_answer = "Agent failed: too many parse errors."
                    break
                
                # Re-prompt with error message
                messages.append({"role": "assistant", "content": response_text})
                messages.append({
                    "role": "user",
                    "content": (
                        f"I couldn't parse your response (attempt {parse_failures}/{max_parse_failures}). "
                        "Please use EXACTLY this format:\n"
                        "Thought: [your reasoning]\n"
                        "Action: [tool name]\n"
                        "Action Input: [tool input]"
                    )
                })
                continue
            
            # Check for loops
            if self._detect_loop(parsed['action'], parsed['action_input']):
                step.error = "Loop detected: same action repeated"
                step.duration_ms = (time.time() - step_start) * 1000
                trace.add_step(step)
                
                if self.verbose:
                    print(f"  LOOP DETECTED: {parsed['action']}({parsed['action_input'][:50]})")
                
                messages.append({"role": "assistant", "content": response_text})
                messages.append({
                    "role": "user",
                    "content": (
                        "You are repeating the same action. This suggests either:\n"
                        "1. The tool isn't giving useful results — try a different tool or approach\n"
                        "2. You already have the answer — provide your Final Answer\n"
                        "Please either try a different approach or give your Final Answer."
                    )
                })
                continue
            
            # Execute tool with retry
            step.action = parsed['action']
            step.action_input = parsed['action_input']
            
            if self.verbose:
                print(f"  Action: {step.action}")
                print(f"  Input: {step.action_input}")
            
            observation = self._execute_with_retry(step.action, step.action_input)
            step.observation = observation
            self._action_history.append((step.action, step.action_input))
            
            if self.verbose:
                print(f"  Observation: {observation[:200]}")
            
            step.duration_ms = (time.time() - step_start) * 1000
            trace.add_step(step)
            
            # Add to messages
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        trace.total_duration_ms = (time.time() - run_start) * 1000
        
        if not trace.final_answer:
            trace.final_answer = "Agent reached maximum steps without finding an answer."
        
        return trace.final_answer, trace


print("RobustReActAgent defined.")

In [ ]:
# Run the robust agent and examine the trace
robust_agent = RobustReActAgent(
    tools=[CalculatorTool(), SearchTool(), PythonREPLTool()],
    max_steps=10,
    max_retries=3,
    verbose=True,
)

answer, trace = robust_agent.run(
    "What is the ratio of Germany's GDP to France's GDP? "
    "Search for both values and calculate the ratio."
)

# Display the full trace
trace.display()

---
## 7. Agent Traces & Debugging

When agents fail in production, you need to understand **why**. The trace is your debugging tool.

Key things to log:
- Every thought (reasoning quality)
- Every action + input (tool selection accuracy)
- Every observation (tool output quality)
- Timing (latency bottlenecks)
- Errors (where things go wrong)

In [ ]:
# The trace we already captured gives us everything we need.
# Let's also build a trace analyzer:

class TraceAnalyzer:
    """Analyzes agent traces to identify issues and patterns."""
    
    @staticmethod
    def analyze(trace: AgentTrace) -> Dict[str, Any]:
        """Analyze a trace and return diagnostic information."""
        analysis = {
            "total_steps": len(trace.steps),
            "total_duration_ms": trace.total_duration_ms,
            "avg_step_duration_ms": (
                trace.total_duration_ms / len(trace.steps)
                if trace.steps else 0
            ),
            "tool_calls": trace.total_tool_calls,
            "errors": len(trace.errors),
            "error_rate": len(trace.errors) / max(len(trace.steps), 1),
            "success": trace.final_answer is not None and "failed" not in trace.final_answer.lower(),
        }
        
        # Tool usage distribution
        tool_usage = {}
        for step in trace.steps:
            if step.action:
                tool_usage[step.action] = tool_usage.get(step.action, 0) + 1
        analysis["tool_usage"] = tool_usage
        
        # Identify slow steps
        slow_steps = [
            s.step_number for s in trace.steps
            if s.duration_ms > analysis["avg_step_duration_ms"] * 2
        ]
        analysis["slow_steps"] = slow_steps
        
        # Identify potential issues
        issues = []
        if analysis["error_rate"] > 0.3:
            issues.append("High error rate — check tool reliability")
        if analysis["total_steps"] >= 8:
            issues.append("Many steps — consider better planning or tool design")
        if len(tool_usage) == 1 and analysis["tool_calls"] > 3:
            issues.append("Only one tool used repeatedly — possible loop or wrong tool selection")
        if not analysis["success"]:
            issues.append("Agent did not succeed — review trace for root cause")
        analysis["issues"] = issues
        
        return analysis
    
    @staticmethod
    def display_analysis(analysis: Dict[str, Any]):
        """Pretty-print trace analysis."""
        print("\nTRACE ANALYSIS")
        print("=" * 40)
        print(f"Steps: {analysis['total_steps']}")
        print(f"Duration: {analysis['total_duration_ms']:.0f}ms")
        print(f"Avg step: {analysis['avg_step_duration_ms']:.0f}ms")
        print(f"Tool calls: {analysis['tool_calls']}")
        print(f"Errors: {analysis['errors']} ({analysis['error_rate']:.0%})")
        print(f"Success: {analysis['success']}")
        print(f"Tool usage: {analysis['tool_usage']}")
        if analysis['slow_steps']:
            print(f"Slow steps: {analysis['slow_steps']}")
        if analysis['issues']:
            print(f"\nIssues detected:")
            for issue in analysis['issues']:
                print(f"  - {issue}")
        else:
            print("\nNo issues detected.")


# Analyze our previous trace
analysis = TraceAnalyzer.analyze(trace)
TraceAnalyzer.display_analysis(analysis)

---
## 8. "Why Does This Work?"

### Why interleave thinking and acting?

**Thinking alone** (chain-of-thought) can reason but has no way to verify facts or interact with the world. It *hallucinates* because it can't ground its reasoning in reality.

**Acting alone** (tool use without reasoning) can get real data but has no way to *plan* what to do with it. It's like having a calculator but no idea what to calculate.

**ReAct combines both**: reasoning creates plans and interprets results, acting grounds reasoning in real observations. Each step of reasoning is informed by real tool outputs, and each tool call is guided by a reasoning plan.

### ReAct vs Chain-of-Thought vs Act-only

| Approach | Grounded? | Plans? | Handles novelty? | Error recovery? |
|----------|-----------|--------|-------------------|------------------|
| **CoT only** | No (hallucination risk) | Yes | Limited | No |
| **Act only** | Yes | No | No | No |
| **ReAct** | Yes | Yes | Yes | Yes (re-plan) |

### What's the biggest failure mode?

1. **Tool selection errors**: The agent picks the wrong tool (e.g., Calculator when it should Search). This cascades — wrong observations lead to wrong reasoning.

2. **Hallucinated tool calls**: The agent invents tools that don't exist or calls tools with impossible arguments. This is worse with weaker models.

3. **Context window overflow**: Each step adds tokens. After 5-10 steps, the context is full. Old observations get pushed out, and the agent loses track of its progress.

4. **Infinite loops**: The agent repeats the same action hoping for a different result. This is why loop detection is critical.

### Why no frameworks in interviews?

Frameworks (LangChain, CrewAI, etc.) hide the complexity that interviewers want to evaluate:

- **Can you design the tool interface?** Frameworks give you a `BaseTool` class.
- **Can you handle parsing failures?** Frameworks do this internally.
- **Can you manage context windows?** Frameworks handle message history.
- **Do you understand the failure modes?** Frameworks abstract them away.

Building from scratch demonstrates that you understand every layer. That's the difference between someone who *uses* agents and someone who *builds* agents.

---
## Interview Question Bank

*These are representative questions modeled on those at Anthropic, OpenAI, Google DeepMind, and Meta FAIR for senior/principal ML research roles in agentic AI.*

---

### Q1: "Build a ReAct agent from scratch in 30 minutes" -- LIVE CODING

**What this tests**: Can you go from concept to working code under pressure? This is the single most common coding question for agent-focused roles.

**This is a REAL interview question at Anthropic, OpenAI, and Google. You code it live.**

**Good answer** (hire): Working agent with tool calling that completes a multi-step task in 30 minutes. Clean separation of thought/action/observation. Shows understanding of the loop.

**Great answer** (strong hire): All of the above, plus:
- Error handling for malformed LLM outputs (regex fallback, retry with re-prompting)
- Retry logic with exponential backoff for API failures
- Structured output parsing (not just string splitting)
- Max iteration guard to prevent infinite loops
- Clean trace/logging for debugging

**Red flag**: Cannot write the basic loop without heavy hints. Confuses ReAct with chain-of-thought. Does not handle the case where the LLM output does not match the expected format.

**Follow-up 1**: "Now add a tool that requires authentication. How do you handle secrets?"
- Good: environment variables, secrets manager, never in code
- Great: discusses tool-level auth scoping, token refresh, audit logging of which tools accessed which credentials

**Follow-up 2**: "Your agent is stuck in a loop calling the same tool with the same arguments. How do you detect and fix this?"
- Good: track action history, detect duplicates, break after N repeats
- Great: discusses why this happens (LLM has no memory of failure within the prompt), how to inject "you already tried this and it failed" into the context, the connection to the exploration-exploitation tradeoff

---

### Q2: "Compare ReAct, function calling, and code-generation approaches to tool use"

**What this tests**: Breadth of knowledge about agent architectures. Do you know the landscape, or just one approach?

**Good answer**: Describes each approach clearly:
- ReAct: interleaved reasoning and action in natural language
- Function calling: structured JSON schema, model outputs function name + args
- Code generation: model writes executable code (e.g., Python) that calls APIs

**Great answer**: Discusses the trade-offs along specific axes:

| Dimension | ReAct | Function Calling | Code Generation |
|-----------|-------|-----------------|-----------------|
| Reliability | Medium (free-form parsing) | High (schema validation) | Medium (code can crash) |
| Latency | Higher (verbose reasoning) | Lower (structured) | Variable (execution time) |
| Flexibility | High (any reasoning) | Low (predefined schema) | Very high (arbitrary logic) |
| Debuggability | Good (human-readable trace) | Good (structured logs) | Hard (runtime errors) |
| When to use | Research, complex reasoning | Production APIs, reliability-critical | Data analysis, complex logic |

**Red flag**: Only knows one approach. Cannot discuss when you would choose one over another.

---

### Q3: "What are the failure modes of LLM agents in production?"

**What this tests**: Production experience. Have you actually deployed agents, or just read papers?

**Good answer**: Tool errors, hallucinated actions (calling tools that do not exist), wrong tool selection, incorrect arguments.

**Great answer**: Covers the full taxonomy of production failures:
1. **Context window overflow**: Agent accumulates too many observations, loses early context, starts contradicting itself
2. **Cost explosions**: Unbounded loops consume thousands of dollars in API calls before anyone notices
3. **Safety failures**: Agent takes harmful real-world actions (deletes files, sends emails, makes purchases) -- the stakes are different from chatbots
4. **Cascading failures**: One bad tool call returns garbage, agent reasons over garbage, subsequent actions are all wrong
5. **Non-determinism**: Same input produces different traces, making debugging nearly impossible without comprehensive logging
6. **Hallucinated tool schemas**: Agent invents tools or arguments that do not exist in the registry
7. **Partial completion**: Agent completes 80% of a task correctly, then silently fails on the last step -- harder to detect than total failure

**Red flag**: Only mentions "hallucination" as a generic answer. No awareness of cost or safety failure modes.

---
## Production Implementation Notes

*What the actual systems look like at frontier labs -- context you need for system design interviews.*

### How Frontier Labs Implement Tool Use

**Anthropic's Claude**:
- Uses XML tool schemas with structured tool results
- Automatic retry on malformed output (the model re-generates if its output does not match the schema)
- Tool results are injected as structured `<tool_result>` blocks, not free-form text
- Claude Code (the CLI agent) is a single main agent loop that can optionally spawn subagents for delegated subtasks -- its internals are not a publicly documented supervisor/worker multi-agent system, so don't present that as fact

**OpenAI's Function Calling**:
- JSON Schema is passed to the model, but plain function calling can still produce invalid or missing arguments -- only Structured Outputs / strict mode adds the schema-conformance guarantee
- Supports parallel function calls (model can request multiple tools in one turn)
- Streaming tool use: tool call arguments are streamed token-by-token, so you can start processing before the full call is complete
- Structured Outputs mode guarantees schema conformance via constrained decoding

**Google's Gemini**:
- Native function calling with automatic schema validation
- Supports "function calling mode" that forces the model to always call a function (no free-text responses)
- Grounding with Google Search as a built-in tool

### Production Guardrails You Must Know

| Guardrail | Why | Implementation |
|-----------|-----|----------------|
| **Rate limiting** | Prevent cost explosions from runaway loops | Token budget per session (e.g., max 50K tokens), max iterations (e.g., 25 steps) |
| **Cost caps** | Prevent $1000 API bills from a single user session | Per-session dollar limit, alert at 80% threshold |
| **Audit logging** | Debug failures, compliance, safety review | Log every LLM call + tool invocation with timestamps, full inputs/outputs |
| **Human-in-the-loop** | High-stakes actions (delete, purchase, send) | Require explicit user confirmation before executing irreversible actions |
| **Sandboxing** | Prevent code execution agents from damaging the host | Docker containers, restricted filesystem access, network isolation |
| **Timeout** | Prevent agents from running indefinitely | Hard timeout per step (30s) and per session (5min) |

### The Uncomfortable Truth About Agent Reliability

> Even GPT-4 and Claude agents fail **20-40%** on complex multi-step tasks (5+ steps with tool use). This is the current state of the art. If an interviewer asks "how reliable are agents?" and you say "very reliable," that is a red flag. The honest answer is "reliable enough for low-stakes tasks with human oversight, not yet reliable enough for fully autonomous high-stakes workflows."

**Benchmark reality check**:
- SWE-bench Verified: best agents went from ~33% (late 2024) to ~70% (mid-2025) to ~80% (early 2026). Note: OpenAI stopped reporting/evaluating SWE-bench Verified in Feb 2026, citing test-quality and contamination concerns.
- WebArena and GAIA: scores rose sharply through 2025-2026 -- check the current leaderboards before interviews.

These numbers move fast; verify against live leaderboards before quoting them.

---
## How This Gets Tested in Interviews

*The meta-knowledge: what interviewers are actually looking for when they ask agent questions.*

### Interview Format by Company

| Company | Format | Duration | What They Want |
|---------|--------|----------|---------------|
| **Anthropic** | Live coding + research discussion | 45 min | Build a working agent. Then discuss: what would break in production? How would you improve it? |
| **OpenAI** | System design + coding | 60 min | Design a complete agent system. Whiteboard the architecture, then code the core loop. |
| **Google DeepMind** | Research presentation + deep dive | 45 min | Present your agent research. Then: "What are the open problems? What would you work on next?" |
| **Meta FAIR** | Paper discussion + implementation | 45 min | Discuss a recent agent paper (they pick it). "What are the limitations? How would you fix them?" |

### The Three Things That Separate Senior from Staff/Principal

1. **Production awareness**: Junior candidates describe the ideal case. Senior candidates immediately discuss failure modes, cost, latency, safety. When you describe an agent, always volunteer: "here is what would break, and here is how I would handle it."

2. **Trade-off reasoning**: Never say "X is better than Y." Always say "X is better than Y *when* Z, but Y wins *when* W." Interviewers are testing whether you can make nuanced engineering decisions, not whether you memorized the right answer.

3. **Research taste**: Can you identify which problems are important vs. solved? The question "what would you work on" is not hypothetical -- it is a test of whether you understand the frontier. Good answers reference specific open problems (e.g., "agent reliability on 10+ step tasks is still below 60% -- I think the key bottleneck is...").

### Signals That Get People Hired

- Builds working code quickly and cleanly (not perfect, but functional)
- Immediately discusses what could go wrong without being asked
- Connects implementation details to research ideas (e.g., "this retry logic is related to the self-correction work in Reflexion")
- Has opinions backed by evidence, not just preferences
- Acknowledges uncertainty honestly ("I am not sure about this, but my intuition is...")

### Signals That Get People Rejected

- Cannot write a basic agent loop without heavy prompting
- Describes only the happy path, never failure modes
- Name-drops papers without understanding them
- Claims agents are "basically solved" or "completely unreliable" (both extremes are wrong)
- Cannot discuss cost/latency trade-offs (suggests they have never deployed anything)

---
## 9. Flashcard Summary

Study these Q&A pairs for interview preparation. Cover the answer and test yourself.

| # | Question | Answer |
|---|----------|--------|
| 1 | What are the three components of an LLM agent? | **LLM** (reasoning engine), **Tools** (external capabilities), **Loop** (observe-think-act cycle) |
| 2 | What does ReAct stand for? | **Re**asoning + **Act**ing. Interleaves chain-of-thought reasoning with tool-use actions. |
| 3 | What is the ReAct format? | Thought → Action → Action Input → Observation → repeat → Final Answer |
| 4 | How does a chatbot differ from an agent? | Chatbot: single LLM call, no tools, stateless. Agent: multi-step loop, uses tools, maintains state. |
| 5 | What is a Tool in agent architecture? | An interface with name, description (for LLM), parameter schema, and execute method. |
| 6 | Why does the tool description matter? | The LLM reads it to decide when/how to use the tool. Bad descriptions → wrong tool selection. |
| 7 | What is function calling vs text-based tool use? | Function calling: LLM outputs structured JSON (reliable). Text-based: LLM outputs free text, parsed with regex (brittle). |
| 8 | What are the three categories of agent failure? | Tool failures (exceptions, timeouts), Parse failures (bad format), Logic failures (loops, wrong tool). |
| 9 | How do you handle tool failures? | Retry with exponential backoff, fallback to alternative tools, timeout enforcement. |
| 10 | How do you detect infinite loops in agents? | Track action history, detect repeated (action, input) pairs in recent steps. |
| 11 | Why interleave reasoning and acting? | Reasoning alone hallucinates (ungrounded). Acting alone lacks planning. Together: grounded + planned. |
| 12 | What is an agent trace? | A complete log of every step: thought, action, observation, timing, errors. Essential for debugging. |
| 13 | What happens when context window fills up? | Old observations are pushed out, agent loses track of progress. Solution: working memory management. |
| 14 | Why build agents from scratch in interviews? | Frameworks hide understanding. Building from scratch proves you understand every layer (tools, parsing, loop, errors). |
| 15 | What is the key finding of the ReAct paper? | ReAct outperforms CoT-only and Act-only on knowledge-intensive tasks by grounding reasoning in real observations. |

---
## 10. Paper Guide: ReAct (Yao et al. 2022)

**Paper**: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)  
**Venue**: ICLR 2023  
**Reading time**: ~45 minutes  

### Section-by-Section Guide

**Abstract & Introduction (5 min)**
- Key claim: interleaving reasoning and acting improves both
- Prior work treats them separately (CoT for reasoning, tool-use for acting)
- ReAct shows they are synergistic

**Section 2: ReAct Framework (10 min)**
- The Thought/Action/Observation format
- How thoughts serve as "internal reasoning" that guides actions
- How observations ground thoughts in reality
- Key: the LLM generates both thoughts AND actions — this is the core innovation

**Section 3: Knowledge-Intensive Tasks (10 min)** — READ CAREFULLY
- HotpotQA and FEVER benchmarks
- ReAct vs CoT vs Act-only vs CoT+Self-Consistency
- Table 1: results are mixed -- on HotpotQA, CoT slightly beats ReAct (29.4 vs 27.4 EM); ReAct wins on FEVER; the best configuration combines ReAct with CoT-SC
- Key finding: ReAct hallucinates less because it can verify via search

**Section 4: Decision-Making Tasks (10 min)**
- ALFWorld and WebShop benchmarks
- ReAct significantly outperforms Act-only (planning matters)
- Table 3: ReAct (best of 6 prompts) reaches 71% success on ALFWorld vs 45% for the Act-only baseline (BUTLER, the imitation-learning baseline: 37%)

**Section 5: Analysis (5 min)** — IMPORTANT for interviews
- Error analysis: what goes wrong?
- Reasoning errors vs action errors vs hallucination
- ReAct + CoT-SC (self-consistency) gets even better

**Section 6: Related Work (skim)**
- Connections to chain-of-thought, inner monologue, and tool-augmented LMs

### Key Experiments to Know

1. **HotpotQA**: ReAct achieves competitive accuracy with far fewer hallucinations
2. **ALFWorld**: ReAct nearly doubles success rate over act-only baselines
3. **Error analysis**: ReAct's failures are mostly retrieval failures (bad search results), not reasoning failures

### Limitations (interview discussion points)

- Requires good prompting — format is fragile without function calling
- Each step adds tokens — context window pressure
- Tool quality limits agent quality — garbage-in-garbage-out
- No learning across episodes (each run starts fresh)

### Interview-Ready Soundbite

> "ReAct showed that reasoning and acting are synergistic: reasoning helps the agent plan and interpret results, while acting grounds the reasoning in real observations. This eliminates the hallucination problem that pure chain-of-thought suffers from, and the directionless exploration that tool-only agents face."

---
## 11. Additional Paper Reference

### Building Effective Agents (Anthropic, 2024)

**Link**: [https://www.anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents)

This is THE reference for production agent design. It identifies **5 agentic patterns**:

1. **Prompt Chaining**: Break a task into fixed sequence of LLM calls, each processing the previous output. Simplest pattern -- use when the workflow is predictable.
2. **Routing**: Classify the input and route to a specialized handler. Use when you have distinct task types requiring different approaches.
3. **Parallelization**: Run multiple LLM calls simultaneously and aggregate. Use for tasks with independent subtasks (e.g., evaluating multiple aspects of a document).
4. **Orchestrator-Workers**: A central LLM dynamically decomposes tasks and delegates to worker LLMs. Use for complex tasks where subtasks can't be predicted in advance.
5. **Evaluator-Optimizer**: One LLM generates output, another evaluates and provides feedback in a loop. Use when you have clear quality criteria and iterative refinement helps.

**Key principle**: "Start with the simplest architecture that works. Don't over-engineer."

### Interview Relevance

If asked "how would you design an agent for X?", start by identifying which of these 5 patterns fits. This shows structured thinking and awareness of the current best practices from Anthropic's own research.

In [ ]:
print("Notebook 13 complete!")
print("\nKey takeaways:")
print("1. Agent = LLM + Tools + Loop (not just a single LLM call)")
print("2. ReAct interleaves thinking and acting for grounded reasoning")
print("3. Function calling is more reliable than text parsing")
print("4. Error handling is what separates demos from production")
print("5. Always trace and log — debugging agents without traces is impossible")
print("\nNext: Notebook 14 — Planning & Memory for Agents")